# GNN Explorer Showcase

这个 notebook 只展示仓库里目前比较成熟、适合直接演示的能力：

- `GraphVisualizer`：双视图图结构可视化
- `GraphVisualizer` 的子图抽样：适合大图局部展示
- `GraphEditor`：在 notebook 里交互式编辑图
- `GNNVisualizer`：用预计算的中间表示展示 GNN 可视化交互

如果 widget 没有正常显示，先在仓库根目录执行一次 `npm run build`。

In [ ]:
import sys
from pathlib import Path

from IPython.display import display, Markdown

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from gnn_exp import GraphVisualizer, GraphEditor, GNNVisualizer

## 1. 小图双视图展示

这里直接用 Karate Club 数据集，适合快速展示 node-link view 和 matrix view 的联动。

In [ ]:
karate_path = repo_root / "test_data" / "karate_dataset.json"

graph_vis = GraphVisualizer()
graph_vis.add_data(str(karate_path))
graph_vis

## 2. 大图子图抽样

完整的 Twitch 图比较大，演示时更适合抽取一个局部子图。下面展示两个 hub 节点的一跳邻域。

In [ ]:
twitch_path = repo_root / "test_data" / "twitch.json"

subgraph_vis = GraphVisualizer()
subgraph_vis.add_data(str(twitch_path))
subgraph_vis.multiple_subgraph_hoop_visualizer(hubNodes=[0, 1], hoopNum=1)
subgraph_vis

## 3. 交互式图编辑

这个组件适合现场展示“修改图结构并立即查看结果”的工作流。编辑完成后可以调用 `export_data()` 或 `export_data_to_json(...)` 导出。

In [ ]:
editor = GraphEditor()
editor.add_data(str(karate_path.relative_to(repo_root)))
editor

## 4. GNN 中间表示可视化

为了让这个 notebook 更容易复现，这里不依赖 PyG 现场跑模型，而是直接构造一份前端已经支持的数据格式，展示 `GNNVisualizer` 的成熟交互。

In [ ]:
graph_data = {
    "x": [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [0.2, 0.8]],
    "edge_index": [[0, 1, 1, 2, 2, 3], [1, 0, 2, 1, 3, 2]],
    "y": [0, 1, 0, 1],
}

intm_data = {
    "act0": [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [0.2, 0.8]],
    "act1": [[0.9, 0.1], [0.3, 0.7], [0.8, 0.2], [0.4, 0.6]],
    "softmax": [[0.92, 0.08], [0.12, 0.88], [0.76, 0.24], [0.35, 0.65]],
}

model_info = {
    "conv1": {
        "type": "GCNConv",
        "weight": [[1.0, 0.0], [0.0, 1.0]],
        "bias": [0.0, 0.0],
    },
    "classifier": {
        "type": "Linear",
        "weight": [[1.0, 0.0], [0.0, 1.0]],
        "bias": [0.0, 0.0],
    },
}

gnn_vis = GNNVisualizer(
    graphData=graph_data,
    intmData=intm_data,
    modelInfo=model_info,
    queries=[[0, 2]],
    subgraphSample=False,
    mode="node",
)

gnn_vis

## 5. 演示建议

比较自然的一套 demo 顺序是：

1. 先展示 Karate 小图的双视图联动。
2. 再切到 Twitch 局部子图，说明大图会先抽样后展示。
3. 然后展示编辑器，现场加点或改边。
4. 最后用 `GNNVisualizer` 展示一层 GNN 到分类输出的中间表示。